In [ ]:
import pandas as pd
import random
from datetime import datetime, timedelta


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.91.230.12:8501



In [ ]:
data = []
timestamp = datetime.now()
for _ in range(30):
    data.append({
        "timestamp": timestamp.strftime("%Y-%m-%d %H:%M:%S"),
        "heart_rate": random.randint(60, 120),
        "spo2": random.randint(92, 100),
        "temperature": round(random.uniform(36.0, 38.5), 1),
        "stress_label": random.choice(["Normal", "High Stress"])
    })
    timestamp += timedelta(seconds=2)

In [ ]:
df = pd.DataFrame(data)

In [ ]:
file_path = "stored_data.csv"
df.to_csv(file_path, index=False)

file_path

'stored_data.csv'

In [ ]:
from google.colab import files
files.download('stored_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("stored_data.csv")

print("\nDataset Preview:")
print(df.head())


Dataset Preview:
             timestamp  heart_rate  spo2  temperature stress_label
0  2026-01-01 07:43:51         120    99         37.5  High Stress
1  2026-01-01 07:43:53          77    92         36.5  High Stress
2  2026-01-01 07:43:55          94   100         38.4       Normal
3  2026-01-01 07:43:57         117    99         37.0       Normal
4  2026-01-01 07:43:59          92    95         38.3       Normal


In [ ]:
df["stress_label"] = df["stress_label"].map({"Normal": 0, "High Stress": 1})

In [ ]:
X = df[["heart_rate", "spo2", "temperature"]]
y = df["stress_label"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, random_state=42)

In [ ]:
y_pred = rf_model.predict(X_test)
print("\nRandom Forest Classification Report:")
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))


Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.75      0.67         4
           1       0.00      0.00      0.00         2

    accuracy                           0.50         6
   macro avg       0.30      0.38      0.33         6
weighted avg       0.40      0.50      0.44         6

Accuracy: 0.5


In [ ]:
joblib.dump(rf_model, "stress_detector_model.pkl")
print("\nRandom Forest Model Saved as: stress_detector_model.pkl")


Random Forest Model Saved as: stress_detector_model.pkl


In [ ]:
hr_data = df["heart_rate"].values.reshape(-1, 1)
mean = hr_data.mean()
std = hr_data.std()
hr_scaled = (hr_data - mean) / std

In [ ]:
def create_sequences(data, steps=5):
    X, y = [], []
    for i in range(len(data) - steps):
        X.append(data[i:i+steps])
        y.append(data[i+steps])
    return np.array(X), np.array(y)

seq_length = 5
X_seq, y_seq = create_sequences(hr_scaled, seq_length)

X_seq = X_seq.reshape((X_seq.shape[0], X_seq.shape[1], 1))

In [ ]:
train_size = int(len(X_seq) * 0.8)
X_train_seq, X_test_seq = X_seq[:train_size], X_seq[train_size:]
y_train_seq, y_test_seq = y_seq[:train_size], y_seq[train_size:]

In [ ]:
lstm_model = Sequential([
LSTM(64, activation="relu", input_shape=(seq_length, 1)),
Dense(32, activation='relu'),
Dense(1)
])

lstm_model.compile(optimizer="adam", loss="mse")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
early_stop = EarlyStopping(monitor="loss", patience=3)
lstm_model.fit(X_train_seq, y_train_seq, epochs=25, batch_size=8, callbacks=[early_stop])

Epoch 1/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 1.0033
Epoch 2/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 1.0413
Epoch 3/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.9351
Epoch 4/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 1.0044
Epoch 5/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.9657
Epoch 6/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.9942
Epoch 7/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.8605
Epoch 8/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.8765
Epoch 9/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.8785
Epoch 10/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.9258
Epoch 11/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.9195
Epoch 12/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.8892
Epoch 13/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.8686
Epoch 14/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.8003
Epoch 15/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.9538
Epoch 16/25
3/3 ━━━━━━━━━━━━━━━━━━

In [ ]:
lstm_model.save("heart_rate_forecasting_model.h5")
print("\nLSTM Model Saved as: heart_rate_forecasting_model.h5")


LSTM Model Saved as: heart_rate_forecasting_model.h5


In [ ]:
last_input = hr_scaled[-seq_length:]
forecast = []

for _ in range(10):
    pred = lstm_model.predict(last_input.reshape(1, seq_length, 1))[0][0]
    forecast.append(pred)
    last_input = np.append(last_input[1:], pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 441ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


In [ ]:
forecast_real = (np.array(forecast) * std) + mean

print("\nNext 10 Predicted Heart Rate Values:")
print(forecast_real)


Next 10 Predicted Heart Rate Values:
[82.70759491 82.43579928 84.92555074 87.61291811 86.26736761 88.28744354
 87.55435059 87.02970099 86.76901549 86.91156057]


In [ ]:
import datetime

In [ ]:
last_timestamp = pd.to_datetime(df["timestamp"].iloc[-1])
last_spo2 = df["spo2"].iloc[-1]
last_temp = df["temperature"].iloc[-1]
future_predictions = []
for i, hr in enumerate(forecast_real):
  next_time = last_timestamp + datetime.timedelta(seconds=(i+1)*2)
  spo2_pred = last_spo2 + np.random.uniform(-0.3, 0.3)
  temp_pred = last_temp + np.random.uniform(-0.05, 0.05)
  feature_row = np.array([[hr, spo2_pred, temp_pred]])
  stress_pred = rf_model.predict(feature_row)[0]
  stress_label = "Normal" if stress_pred == 0 else "High Stress"
  future_predictions.append({
  "timestamp": next_time,
  "predicted_heart_rate": round(hr, 2),
  "predicted_spo2": round(spo2_pred, 2),
  "predicted_temperature": round(temp_pred, 2),
  "predicted_stress_state": stress_label
  })

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local

In [ ]:
future_df = pd.DataFrame(future_predictions)
print("\n FUTURE PREDICTION")
print(future_df)


 FUTURE PREDICTION
            timestamp  predicted_heart_rate  predicted_spo2  \
0 2026-01-01 07:44:51                 82.71           95.22   
1 2026-01-01 07:44:53                 82.44           95.17   
2 2026-01-01 07:44:55                 84.93           95.04   
3 2026-01-01 07:44:57                 87.61           95.01   
4 2026-01-01 07:44:59                 86.27           95.22   
5 2026-01-01 07:45:01                 88.29           94.76   
6 2026-01-01 07:45:03                 87.55           94.82   
7 2026-01-01 07:45:05                 87.03           94.90   
8 2026-01-01 07:45:07                 86.77           94.76   
9 2026-01-01 07:45:09                 86.91           95.17   

   predicted_temperature predicted_stress_state  
0                  36.67            High Stress  
1                  36.70            High Stress  
2                  36.72            High Stress  
3                  36.68                 Normal  
4                  36.68            

In [ ]:
future_df.to_csv("future_predictions.csv", index=False)
print("\nPredictions saved as: future_predictions.csv")


Predictions saved as: future_predictions.csv


In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
df_future = pd.read_csv("future_predictions.csv")
refresh_button = widgets.Button(description="Refresh Predictions", style={'button_color': '#4CAF50'})
label_title = widgets.HTML("<h3>Human Digital Twin - Prediction Dashboard</h3>")
output_area = widgets.Output()
def update_dashboard(change=None):
    with output_area:
        clear_output()
        display(df_future)
        plt.figure(figsize=(8, 4))
        plt.plot(df_future['timestamp'], df_future['predicted_heart_rate'], marker='o')
        plt.xlabel("Time")
        plt.ylabel("Predicted Heart Rate (bpm)")
        plt.title("Heart Rate Forecast")
        plt.xticks(rotation=45)
        plt.grid(True)
        plt.show()
        stress_counts = df_future['predicted_stress_state'].value_counts()
        normal = stress_counts.get("Normal", 0)
        stress = stress_counts.get("High Stress", 0)
        if stress > 0:
            status_html = f"<p style='color:red; font-size:18px;'>⚠ Stress Detected in {stress} Future Points</p>"
        else:
            status_html = "<p style='color:green; font-size:18px;'>✔ All Future Predictions Normal</p>"
        display(widgets.HTML(status_html))
refresh_button.on_click(update_dashboard)
dashboard = widgets.VBox([
    label_title,
    refresh_button,
    output_area
])
display(dashboard)
update_dashboard()


In [ ]:
import time
for _ in range(5):
    update_dashboard()
    time.sleep(5)

In [ ]:
!pip install ipywidgets matplotlib pandas numpy joblib


In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
rf_model = joblib.load("stress_detector_model.pkl")
lstm_model = load_model("heart_rate_forecasting_model.h5", compile=False)
print("Models loaded successfully")

Models loaded successfully


In [ ]:
import random
import pandas as pd
from datetime import datetime
def generate_live_data():
    return {
        "timestamp": datetime.now(),
        "heart_rate": random.randint(65, 120),
        "spo2": round(random.uniform(94, 99), 1),
        "temperature": round(random.uniform(36.3, 38.2), 2)
    }

In [ ]:
import numpy as np
hr_buffer = []
def predict_state(data):
    global hr_buffer
    hr_buffer.append(data["heart_rate"])
    X = np.array([[data["heart_rate"], data["spo2"], data["temperature"]]])
    stress = rf_model.predict(X)[0]
    stress_label = "High Stress" if stress == 1 else "Normal"
    if len(hr_buffer) >= 5:
        seq = np.array(hr_buffer[-5:]).reshape(1, 5, 1)
        forecast = lstm_model.predict(seq, verbose=0)[0][0]
    else:
        forecast = None
    return stress_label, forecast

In [ ]:
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
rf_model = joblib.load("stress_detector_model.pkl")
lstm_model = load_model("heart_rate_forecasting_model.h5", compile=False)
print("Models loaded successfully")

Models loaded successfully


In [ ]:
!pip install streamlit pyngrok --quiet


In [ ]:
# %%writefile app.py
# import streamlit as st import pandas as pd import random import time st.title("Human Digital Twin – Real-Time Dashboard") # Load your dataset CSV
# df_data = pd.read_csv("Data1.csv") # <-- replace with your CSV filename # Initialize session state variables
# if 'live_data' not in st.session_state:
#   st.session_state.live_data = []
#   if 'running' not in st.session_state:
#     st.session_state.running = False
#     if 'data_index' not in st.session_state:
#       st.session_state.data_index = 0 def predict_state(data): stress = data.get("stress_label", "Normal") hr = data.get("heart_rate", 70) forecast_hr = hr + random.uniform(-3, 6) return stress, round(forecast_hr, 2)  col1, col2 = st.columns(2) with col1: start = st.button("▶ Start Simulation") with col2: stop = st.button("⏹ Stop Simulation") if start: st.session_state.running = True if stop: st.session_state.running = False placeholder = st.empty() while st.session_state.running:  row = df_data.iloc[st.session_state.data_index].to_dict() st.session_state.data_index += 1 if st.session_state.data_index >= len(df_data): st.session_state.data_index = 0 # Predict stress and forecast stress, forecast_hr = predict_state(row) row["stress"] = stress row["forecast_hr"] = forecast_hr st.session_state.live_data.append(row) df_live = pd.DataFrame(st.session_state.live_data[-61:]) with placeholder.container(): st.dataframe(df_live) st.line_chart(df_live[["heart_rate", "forecast_hr"]]) if stress == "High Stress": st.markdown("### ⚠ HIGH STRESS DETECTED", unsafe_allow_html=True) elif stress == "Moderate Stress": st.markdown("### ⚠ MODERATE STRESS", unsafe_allow_html=True) else: st.markdown("### ✔ NORMAL STATE", unsafe_allow_html=True) time.sleep(10)

In [ ]:
# Replace with your ngrok authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "37bf0WzBSqAuHkxSVh0eFNrwRBr_vFA7in8ub9EVZ92LFnA4"

!ngrok authtoken $NGROK_AUTH_TOKEN


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
!kill $(pgrep ngrok)


In [ ]:
from pyngrok import ngrok
import random
import string

# generate a random tunnel name
random_name = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
public_url = ngrok.connect(8501, name=random_name)
print("Streamlit URL:", public_url)


PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://andre-fervid-darline.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}


In [ ]:
import threading
from pyngrok import ngrok

# Function to run Streamlit in a separate thread
def run_streamlit():
    !streamlit run app.py

# Start Streamlit in background thread
threading.Thread(target=run_streamlit).start()

# Open ngrok tunnel
public_url = ngrok.connect(8501)
print("Your Streamlit app is live at:", public_url)

In [ ]:
!pip install streamlit-autorefresh


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 700.8/700.8 kB 11.0 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import random
import time
from datetime import datetime, timedelta

st.title("Human Digital Twin – Real-Time Dashboard")

# Load your dataset CSV
df_data = pd.read_csv("Data1.csv")  # replace with your CSV filename

# Initialize session state variables
if 'live_data' not in st.session_state:
    st.session_state.live_data = []
if 'running' not in st.session_state:
    st.session_state.running = False
if 'data_index' not in st.session_state:
    st.session_state.data_index = 0
if 'current_time' not in st.session_state:
    last_time = pd.to_datetime(df_data['timestamp'].iloc[-1], format="%d-%m-%Y %H:%M")
    st.session_state.current_time = last_time + timedelta(seconds=1)

# Function to slightly randomize measurements
def generate_live_values(row):
    hr = row["heart_rate"] + random.randint(-2, 3)
    spo2 = row["spo2"] + random.randint(-1, 1)
    temp = row["temperature"] + random.uniform(-0.2, 0.2)

    # Update stress based on HR
    if hr > 95:
        stress = "High Stress"
    elif hr > 85:
        stress = "Moderate Stress"
    else:
        stress = "Normal"

    # Forecast heart rate for plotting
    forecast_hr = hr + random.uniform(-3, 5)

    return hr, spo2, round(temp, 1), stress, round(forecast_hr, 2)

# Buttons
col1, col2 = st.columns(2)
with col1:
    if st.button("▶ Start Simulation"):
        st.session_state.running = True
with col2:
    if st.button("⏹ Stop Simulation"):
        st.session_state.running = False

placeholder = st.empty()

# Main loop
while st.session_state.running:
    row = df_data.iloc[st.session_state.data_index].to_dict()
    st.session_state.data_index += 1
    if st.session_state.data_index >= len(df_data):
        st.session_state.data_index = 0

    # Update timestamp
    row["timestamp"] = st.session_state.current_time.strftime("%d-%m-%Y %H:%M:%S")
    st.session_state.current_time += timedelta(seconds=10)

    # Generate live values
    hr, spo2, temp, stress, forecast_hr = generate_live_values(row)
    row["heart_rate"] = hr
    row["spo2"] = spo2
    row["temperature"] = temp
    row["stress"] = stress
    row["forecast_hr"] = forecast_hr

    # Append to live data
    st.session_state.live_data.append(row)
    df_live = pd.DataFrame(st.session_state.live_data[-20:])

    # Display
    with placeholder.container():
        st.dataframe(df_live)
        st.line_chart(df_live[["heart_rate", "forecast_hr"]])
        if stress == "High Stress":
            st.markdown("### ⚠ HIGH STRESS DETECTED", unsafe_allow_html=True)
        elif stress == "Moderate Stress":
            st.markdown("### ⚠ MODERATE STRESS", unsafe_allow_html=True)
        else:
            st.markdown("### ✔ NORMAL STATE", unsafe_allow_html=True)

    time.sleep(2)


Overwriting app.py
